# M7.A3 재정의 — 모델 ② 메뉴 비중 분해 v1 검증 (38차 후속)

> 배경: 담당자 확정 — 산출 구조는 **2모델**: ① 매출 예측(V1-t, 완료) ② **매장별 메뉴 분해**(공통 모델
> 없음 = 매장별로 자기 데이터에서 추정). 재료 리스트업(레시피)은 점주 관리 → recommend는
> "② 메뉴별 예상 수량 × 점주 레시피 BOM + 재고 로직"의 결정론 전개. 본 노트북은 ②의 v1
> (비중 분해)을 파일럿 매장 메뉴×일 실데이터로 검증. 작성: 2026-07-29

🔒 공개 저장소 정책 — 출력 제거 커밋, 본문 수치는 비중·적중률·수량 단위만(매출 절대액 없음).

**설계** — "예측 매출을 메뉴별로 나눌 비중을 무엇으로 추정하나?" 분해 품질을 매출 예측 오차와
격리하기 위해 **당일 실제 총수량 × 추정 비중**으로 평가(최종 오차 = 매출 예측 오차 ⊕ 분해 오차).
평가 fold는 V1-t와 동일(월 단위 walk-forward 5개, test 2026-04 봉인 유지).

**후보(사전 고정, 추가 탐색 금지)** — S0 직전 영업일 비중(참조 하한) · S1 최근 28영업일 합산 비중 ·
S2 같은 요일 최근 4회 합산 비중(비면 S1 fallback) · S3 (S1+S2)/2.
**사전 등록 판정**: 평균 TV(총변동거리, 0=완벽 1=전혀) 최저 후보 채택, S2/S3 vs S1 fold 승수 병기.

## 판정 요약 (TL;DR)

1. **채택 = S1: 최근 28영업일 합산 비중.** 평균 TV **0.276**, top-5 메뉴 적중률 **73.1%**,
   활성 메뉴 수량 MAE 1.8개 — 요일 조건부(S2)·혼합(S3)이 **5 fold 전패**로 열세.
2. **요일 조건부가 진 이유** — 메뉴 132종에 같은 요일 표본 4일은 잡음이 신호를 압도.
   그리고 요일에 따라 변하는 것은 **판매량 수준**(V1-t가 담당)이지 **메뉴 믹스**가 아님 —
   믹스는 요일 불변에 가깝다는 게 데이터의 답. 분해 모델은 단순할수록 강했다.
3. **재개장 직후에도 동작** — regime fold(2026-03) TV 0.304로 악화폭이 작음. 28일 윈도우가
   신메뉴 체제로 자연 적응. 이력에 전무한 신메뉴의 당일 판매 비중은 평균 1.0%(fold 최대 2.0%) —
   구조적으로 못 맞히는 질량이 작음.
4. **함의 — recommend 파이프라인 확정 가능**: `V1-t 예측 매출 × S1 비중 → 메뉴별 예상 수량
   → 점주 레시피(BOM) 전개 + 재고·리드타임 → 재료별 발주 참고치`. 전 단계 결정론(모델 ②는
   집계 통계)이라 서빙 비용·불확실성 추가 없음. 참고치 성격과 신뢰도 배지 동반 노출은 필수.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
_here = Path.cwd()
AI_DIR = next(p for p in [_here.parent, _here, _here / "AI"] if (p / "data_prep").exists())
sys.path.insert(0, str(AI_DIR / "data_prep"))
import preprocess as pp

df = pd.read_parquet(AI_DIR / "data/processed/sales_daily_menu.parquet")
menu = df[(~df.is_service_item) & (df.qty > 0)][["date", "menu_clean", "qty"]]
piv = menu.pivot_table(index="date", columns="menu_clean", values="qty", aggfunc="sum").fillna(0.0)
piv = piv[piv.sum(axis=1) > 0]
print(f"영업일 {len(piv)}일 × 메뉴 {piv.shape[1]}종 | 기간 {piv.index.min():%Y-%m-%d}~{piv.index.max():%Y-%m-%d}")

folds = pp.make_monthly_folds(piv.index)
SEL = folds[:-1]
print("검증 fold:", [f["month"] for f in SEL], "| test 봉인:", folds[-1]["month"])

rows = []
for f in SEL:
    for d in f["val"]:
        hist = piv.loc[piv.index < d]
        if len(hist) < 10:
            continue
        actual = piv.loc[d]
        total = actual.sum()
        ashare = actual / total

        last1 = hist.iloc[-1]
        s0 = last1 / max(last1.sum(), 1)
        last28 = hist.tail(28).sum()
        s1 = last28 / max(last28.sum(), 1)
        dowh = hist[hist.index.dayofweek == d.dayofweek].tail(4).sum()
        s2 = dowh / dowh.sum() if dowh.sum() > 0 else s1
        shares = {"S0_persist": s0, "S1_recent28": s1, "S2_dow4": s2, "S3_blend": (s1 + s2) / 2}

        top5a = set(actual.nlargest(5).index)
        unseen = float(ashare[hist.sum() == 0].sum())   # 이력 전무 신메뉴의 당일 비중
        for name, sh in shares.items():
            qty_hat = sh * total
            act_mask = (actual > 0) | (qty_hat > 0.5)
            rows.append(dict(fold=f["month"], date=d, cand=name,
                             tv=0.5 * float((sh - ashare).abs().sum()),
                             top5=len(top5a & set(sh.nlargest(5).index)) / 5,
                             mae_active=float((qty_hat - actual)[act_mask].abs().mean()),
                             unseen=unseen))
r = pd.DataFrame(rows)

display(r.groupby("cand").agg(TV=("tv", "mean"), top5=("top5", "mean"),
                              qtyMAE_활성=("mae_active", "mean")).round(3))
pv = r.pivot_table(index="fold", columns="cand", values="tv").round(3)
display(pv)
display(r.pivot_table(index="fold", columns="cand", values="top5").round(2))
print(f"신메뉴(이력 전무) 판매 비중 — 평균 {r.groupby('date').unseen.first().mean():.1%}, "
      f"fold 평균 최대 {r.groupby('fold').unseen.mean().max():.1%}")

agg_tv = r.groupby("cand").tv.mean()
print(f"판정: TV 최저 = {agg_tv.idxmin()} | S2 vs S1 승수 {int((pv['S2_dow4'] < pv['S1_recent28']).sum())}/5 "
      f"| S3 vs S1 승수 {int((pv['S3_blend'] < pv['S1_recent28']).sum())}/5 → 채택 {agg_tv.idxmin()}")

### 관찰 — 세부

- **TV 0.276의 의미**: 하루 판매량 질량의 ~72%를 올바른 메뉴에 배분. 발주 참고 용도에서 중요한 건
  상위 메뉴이고, top-5 적중 73%·persistence(S0) 대비 전 지표 우위로 "최근 4주의 믹스가 내일의
  믹스"라는 가정이 이 매장에서 성립.
- **S2(요일 조건부) 전패의 교훈**: 우리 직관(주점이니 금요일 믹스가 다를 것)과 달리, 요일이 바꾸는
  것은 총량이지 구성비가 아니었다. 조건부를 붙일수록 표본만 얇아짐 — 132종 메뉴에 4일 표본.
  모델 ①(V1-t)이 총량의 요일 효과를 이미 처리하므로 역할 분담도 깔끔: **①은 언제·얼마나, ②는
  무엇을**.
- **한계(정직)**: ① 최종 메뉴별 수량 오차는 매출 예측 오차와 합성 — UI에는 "예상"이 아니라
  "참고치"로, 신뢰도 배지·구간과 동반 노출 ② 판매 0 신메뉴는 비중 0 — 신메뉴 출시 직후는 점주
  수동 보정 영역(제품 정책과 일치) ③ 단일 매장 검증 — 타 매장 이식성은 메뉴 데이터 확보 시 재확인.

### 다음 단계

- recommend 계약 v2 설계(담당자 확정 후): 입력에 `menu_sales_history` 추가, 서버 내부에서
  ①×② → BOM 전개 + 재고·리드타임 → 재료별 발주 참고치. `app/model/decompose.py` + `orders.py` 구현.
- spec 반영: model_spec §3·§4(모델 ② 확정 표기), api_spec §8 recommend v2 — docs PR.